source API URL : "https://archive-api.open-meteo.com/v1/archive?latitude=52.52&longitude=13.41&start_date=2023-01-01&end_date=2024-01-01&daily=temperature_2m_max,temperature_2m_min,rain_sum"

JSON Target File Path : "abfss://bronze@datalakestorageaccountname.dfs.core.windows.net/bronze/weather-data/
"

In [0]:
#weather_source_API_URL = "https://archive-api.open-meteo.com/v1/archive?latitude=52.52&longitude=13.41&start_date=2023-01-01&end_date=2024-01-01&daily=temperature_2m_max,temperature_2m_min,rain_sum"

weather_source_API_base_URL = "https://archive-api.open-meteo.com/v1/archive?"
weather_source_API_URL_options = "&daily=temperature_2m_max,temperature_2m_min,rain_sum"

weather_sink_layer_name = 'adbbronze'
weather_sink_storage_account_name = 'adbstorageu'
weather_sink_folder_name = 'adbbronze/weather'

weather_sink_folder_path = f"abfss://{weather_sink_layer_name}@{weather_sink_storage_account_name}.dfs.core.windows.net/{weather_sink_folder_name}"

## In the abover case we are defaulting to the location of Kovilpatti but there are more than 1000 locations in the dataset, we need to modularize the code

In [0]:
import requests
import json
import pandas as pd

In [0]:
geo_location_df = spark.sql("Select latitude,longitude,market_name from adb_rtp.silver.geo_location_silver limit 100")

In [0]:
weather_API_response_list = []
for geo_locations in geo_location_df.collect():
    #print(geo_locations["market_name"],geo_locations["latitude"],geo_locations["longitude"])
    weather_source_API_URL = f"{weather_source_API_base_URL}latitude={geo_locations['latitude']}&longitude={geo_locations['longitude']}&start_date=2023-01-01&end_date=2023-12-31{weather_source_API_URL_options}"
    weather_API_response = requests.get(weather_source_API_URL).json()
    weather_API_response["market_name"] = geo_locations["market_name"]
    weather_API_response_json = json.dumps(weather_API_response)

    if isinstance(weather_API_response,dict):
        weather_API_response_list.append(weather_API_response_json)

weather_data_RDD = sc.parallelize(weather_API_response_list)
weather_data_df = spark.read.json(weather_data_RDD)
weather_data_df.write.mode("overwrite").json(weather_sink_folder_path)